# 03 · E4 Threshold estimators and E5 assumed contamination (ULB)

No training. Uses the RAREGuard_Full scores from notebook 01, so only the threshold changes.

* **E4**: four label-free estimators on identical validation scores: percentile (c = 0.002), EVT (generalised Pareto fitted to the top 10% of scores, extrapolated to the 0.2% tail), median + 3.5 MAD, and Q3 + 1.5 IQR. The label-assisted threshold is added as a reference row; it uses validation labels and is not one of the label-free methods.
* **E5**: the percentile threshold with c in {0.10, 0.15, 0.20, 0.30, 0.50}%.

PR-AUC and ROC-AUC do not change within a seed here, because the scores do not change.

| | |
|---|---|
| Trains models | no |
| Needs | notebook 01 |
| Writes | `results/E4_E5_thresholds/`: `threshold_methods_by_seed.csv`, `assumed_contamination_by_seed.csv`, their `_summary.csv` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import numpy as np, pandas as pd

EXP = "E4_E5_thresholds"
W = cfg["fusion_weights"]
C = cfg["contamination"]

run = rg.Run(cfg, EXP, "03_E4_E5_thresholds.ipynb", "Threshold estimators (E4) and assumed contamination (E5)")
e4_rows, e5_rows = [], []

for seed in cfg["seeds"]:
    z = rg.load_scores(cfg, "E1_components", f"ulb_seed{seed}")
    val_s = rg.weighted_fusion(rg.scores_from_npz(z, "val"), W)
    test_s = rg.weighted_fusion(rg.scores_from_npz(z, "test"), W)

    thresholds = rg.label_free_thresholds(val_s, contamination=C)
    thresholds["Label_Assisted"], _ = rg.label_assisted_threshold(z["y_val"], val_s, cfg)
    for method, thr in thresholds.items():
        if not np.isfinite(thr):
            print(f"seed {seed}: {method} gave no valid threshold, skipped")
            continue
        e4_rows.append({"Seed": seed, "Method": method,
                        "Uses_Validation_Labels": method == "Label_Assisted",
                        **rg.evaluate(z["y_test"], test_s, thr, val_s)})

    for c in cfg["e5_contamination_grid"]:
        e5_rows.append({"Seed": seed, "Contamination": c,
                        **rg.evaluate(z["y_test"], test_s, rg.percentile_threshold(val_s, c), val_s)})

e4, e5 = pd.DataFrame(e4_rows), pd.DataFrame(e5_rows)
extra = ["Val_Alert_Rate", "Test_Alert_Rate", "TP", "FP"]
run.save_csv(e4, "threshold_methods_by_seed.csv")
run.save_csv(rg.mean_sd(e4, ["Method"], rg.METRICS + extra), "threshold_methods_summary.csv")
run.save_csv(e5, "assumed_contamination_by_seed.csv")
run.save_csv(rg.mean_sd(e5, ["Contamination"], rg.METRICS + extra), "assumed_contamination_summary.csv")
run.finish()

In [ ]:
e1 = rg.read_result(cfg, "E1_components", "runs_by_seed.csv")
e1 = e1[(e1.Experiment == "RAREGuard_Full") & (e1.Threshold_Mode == "Fully unsupervised")].set_index("Seed").F1
pct = e4[e4.Method == "Percentile"].set_index("Seed").F1
assert np.allclose(pct.loc[e1.index], e1), "Percentile row should equal E1 RAREGuard_Full"
assert (e5.groupby("Seed").PR_AUC.nunique() == 1).all()
print("Checks passed.")

cols = ["F1_mean", "F1_sd", "Precision_mean", "Recall_mean", "Test_Alert_Rate_mean"]
display(rg.read_result(cfg, EXP, "threshold_methods_summary.csv").set_index("Method")[cols].round(4))
display(rg.read_result(cfg, EXP, "assumed_contamination_summary.csv").set_index("Contamination")[cols].round(4))